In [ ]:
import xarray
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
import glob
import netCDF4 as nc


In [16]:
import xarray as xr
import pandas as pd
import numpy as np

# NetCDF path (relative to this notebook)
nc_path = "XRO_indices_oras5.nc"

# Load with time decoding
obs_ds = xr.open_dataset(nc_path, decode_times=True)

# Variables that are 1D along time (exclude helper coords like 'month')
vars_1d = [
    name for name, da in obs_ds.data_vars.items()
    if da.ndim == 1 and "time" in da.dims and name != "month"
]

# Define train/test ranges (edit these as needed)
train_start, train_end = "1979-01", "2022-12"
test_start, test_end = "2023-01", None  # None = to latest


def _to_df(ds_slice: xr.Dataset) -> pd.DataFrame:
    sub = ds_slice[vars_1d]
    df = sub.to_dataframe()
    idx = df.index
    try:
        if hasattr(idx, "to_datetimeindex"):
            idx = idx.to_datetimeindex()
        else:
            idx = pd.to_datetime(idx)
    except Exception:
        pass
    df.index = idx
    return df.sort_index()

# Build DataFrames
df_all = _to_df(obs_ds)
train_ds = obs_ds.sel(time=slice(train_start, train_end))
if test_end is None:
    test_ds = obs_ds.sel(time=slice(test_start, None))
else:
    test_ds = obs_ds.sel(time=slice(test_start, test_end))

df_train = _to_df(train_ds)
df_test = _to_df(test_ds)

# Save CSVs
csv_train = f"XRO_indices_oras5_train.csv"
csv_test = f"XRO_indices_oras5_test.csv"

df_train.to_csv(csv_train)
df_test.to_csv(csv_test)

print(f"Observed time span: {df_all.index.min()} → {df_all.index.max()} ({len(df_all)} records)")
print(f"Train: {df_train.index.min()} → {df_train.index.max()} ({len(df_train)}) -> {csv_train}")
print(f"Test:  {df_test.index.min()} → {df_test.index.max()} ({len(df_test)}) -> {csv_test}")

display(df_train.head())
display(df_test.head())


Observed time span: 1979-01-01 00:00:00 → 2024-12-01 00:00:00 (552 records)
Train: 1979-01-01 00:00:00 → 2022-12-01 00:00:00 (528) -> XRO_indices_oras5_train.csv
Test:  2023-01-01 00:00:00 → 2024-12-01 00:00:00 (24) -> XRO_indices_oras5_test.csv


,Nino34,WWV,NPMM,SPMM,IOB,IOD,SIOD,TNA,ATL3,SASD,month
time,,,,,,,,,,,
1979-01-01,-0.142279,11.066086,-0.253452,0.506946,0.187517,0.431726,0.311924,0.233285,-0.187185,0.251324,1
1979-02-01,0.026295,10.962595,-0.611901,0.188063,0.134413,0.070864,0.127076,0.596445,-0.239818,0.343103,2
1979-03-01,-0.082657,5.943167,-0.574011,-0.174663,0.081334,0.044977,0.196739,0.289287,-0.245982,0.349476,3
1979-04-01,0.064349,2.831954,-0.400757,0.093044,0.018981,-0.029559,0.124406,0.442119,-0.040786,0.268618,4
1979-05-01,0.037303,2.936459,-0.442813,0.361989,-0.051639,0.040804,0.118277,0.462302,-0.009008,0.138890,5


,Nino34,WWV,NPMM,SPMM,IOB,IOD,SIOD,TNA,ATL3,SASD,month
time,,,,,,,,,,,
2023-01-01,-0.798631,8.668407,-0.082319,0.180341,-0.477806,0.080787,-0.375966,-0.122710,0.297713,0.898278,1
2023-02-01,-0.601924,6.891833,-0.396517,0.569226,-0.556244,0.186055,-0.271316,-0.213235,0.175474,0.723040,2
2023-03-01,-0.110391,9.779562,-0.362537,0.760076,-0.350446,0.390304,-0.149755,0.170063,-0.079198,0.671144,3
2023-04-01,0.048563,10.238829,-0.526471,1.091174,-0.180337,0.113291,0.040893,0.377102,0.096716,0.217910,4
2023-05-01,0.425916,8.857431,-0.494030,0.434880,-0.163334,-0.173374,0.224030,0.542392,0.290811,0.218350,5
